In [ ]:
# ============================================================
# Limpieza de reviews de Google Places API
# Versión: convertir emojis a texto
# Proyecto: Geomarketing
# Archivo de entrada: reviews_output.csv
# Archivo de salida: reviews_clean_emoji_text.csv
# Reporte: cleaning_report_emoji_text.txt
# ============================================================

import pandas as pd
import re
import unicodedata
import os
from datetime import datetime

# Intentar importar librería emoji
# Si no está instalada, el script seguirá funcionando,
# pero reemplazará emojis por el token [emoji].
try:
    import emoji
    EMOJI_LIBRARY_AVAILABLE = True
except ImportError:
    EMOJI_LIBRARY_AVAILABLE = False


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

INPUT_FILE = "reviews_output.csv"

OUTPUT_FILE = "reviews_clean_emoji_text.csv"
REPORT_FILE = "cleaning_report_emoji_text.txt"

USE_TIMESTAMP_OUTPUT = False

REMOVE_ACCENTS = False

# En esta versión vamos a convertir emojis a texto
KEEP_EMOJIS = False
REMOVE_EMOJIS = False
CONVERT_EMOJIS_TO_TEXT = True

SHORT_REVIEW_WORD_THRESHOLD = 3
DROP_EMPTY_CLEAN_TEXT = False

ENCODING = "utf-8-sig"


# ============================================================
# COLUMNAS ESPERADAS
# ============================================================

REQUIRED_COLUMNS = [
    "place_id",
    "name",
    "rating",
    "user_ratings_total",
    "review_text",
    "review_rating"
]

NUMERIC_COLUMNS = [
    "source_row",
    "rating",
    "user_ratings_total",
    "review_rating"
]


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def print_separator():
    print("-" * 70)


def generate_output_names():
    if not USE_TIMESTAMP_OUTPUT:
        return OUTPUT_FILE, REPORT_FILE

    timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
    output_file = f"reviews_clean_emoji_text_{timestamp}.csv"
    report_file = f"cleaning_report_emoji_text_{timestamp}.txt"

    return output_file, report_file


def warn_if_output_exists(output_file, report_file):
    if os.path.exists(output_file):
        print(f"Advertencia: el archivo '{output_file}' ya existe y será reemplazado.")

    if os.path.exists(report_file):
        print(f"Advertencia: el archivo '{report_file}' ya existe y será reemplazado.")


def load_dataset(input_file):
    if not os.path.exists(input_file):
        print(f"Error: no se encontró el archivo '{input_file}'.")
        print("Verifica que esté en el mismo directorio que este script.")
        return None

    try:
        df = pd.read_csv(input_file, encoding=ENCODING)
        print(f"Archivo leído correctamente: {input_file}")
        return df

    except UnicodeDecodeError:
        print("Error con utf-8-sig. Intentando leer con utf-8...")
        try:
            df = pd.read_csv(input_file, encoding="utf-8")
            print(f"Archivo leído correctamente con utf-8: {input_file}")
            return df
        except Exception as e:
            print("No se pudo leer el archivo.")
            print(f"Detalle del error: {e}")
            return None

    except Exception as e:
        print("Error al leer el archivo.")
        print(f"Detalle del error: {e}")
        return None


def validate_required_columns(df):
    missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]

    if missing_columns:
        print("Error: faltan columnas obligatorias:")
        for col in missing_columns:
            print(f"- {col}")
        return False

    print("Validación correcta: todas las columnas obligatorias existen.")
    return True


def count_empty_review_text(df):
    return (
        df["review_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )


def show_initial_review(df):
    print_separator()
    print("REVISIÓN INICIAL DEL DATASET")
    print_separator()

    print(f"Número inicial de filas: {df.shape[0]}")
    print(f"Número inicial de columnas: {df.shape[1]}")

    print("\nColumnas:")
    for col in df.columns:
        print(f"- {col}")

    print("\nConteo de valores nulos por columna:")
    print(df.isnull().sum())

    print(f"\nReviews vacíos, nulos o solo espacios: {count_empty_review_text(df)}")
    print(f"Negocios únicos usando place_id: {df['place_id'].nunique(dropna=True)}")
    print(f"Duplicados exactos antes de limpiar: {df.duplicated().sum()}")

    subset_duplicates = df.duplicated(
        subset=["place_id", "review_text", "review_rating"]
    ).sum()

    print(f"Duplicados por place_id + review_text + review_rating: {subset_duplicates}")


def has_valid_review_text(value):
    if pd.isna(value):
        return False

    return str(value).strip() != ""


def remove_accents_from_text(text):
    normalized = unicodedata.normalize("NFD", text)
    return "".join(
        char for char in normalized
        if unicodedata.category(char) != "Mn"
    )


def is_emoji_char(char):
    codepoint = ord(char)

    emoji_ranges = [
        (0x1F300, 0x1F5FF),
        (0x1F600, 0x1F64F),
        (0x1F680, 0x1F6FF),
        (0x1F700, 0x1F77F),
        (0x1F780, 0x1F7FF),
        (0x1F800, 0x1F8FF),
        (0x1F900, 0x1F9FF),
        (0x1FA70, 0x1FAFF),
        (0x2600, 0x26FF),
        (0x2700, 0x27BF),
    ]

    return any(start <= codepoint <= end for start, end in emoji_ranges)


def count_emojis(text):
    if pd.isna(text):
        return 0

    text = str(text)

    if EMOJI_LIBRARY_AVAILABLE:
        return emoji.emoji_count(text)

    return sum(1 for char in text if is_emoji_char(char))


def contains_emoji(text):
    return count_emojis(text) > 0


def remove_urls(text):
    url_pattern = r"(https?://\S+|www\.\S+)"
    return re.sub(url_pattern, " ", text, flags=re.IGNORECASE)


def contains_url(text):
    if pd.isna(text):
        return False

    text = str(text)
    url_pattern = r"(https?://\S+|www\.\S+)"

    return bool(re.search(url_pattern, text, flags=re.IGNORECASE))


def remove_control_characters(text):
    return "".join(
        char for char in text
        if unicodedata.category(char)[0] != "C" or char in "\n\t"
    )


def reduce_repeated_punctuation(text):
    text = re.sub(r"!{2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    text = re.sub(r"\.{2,}", ".", text)

    return text


def convert_emojis_to_text(text):
    """
    Convierte emojis a texto en español usando un marcador especial.

    Ejemplo:
    😡 -> [emoji_cara_enojada]

    Esto evita que la descripción del emoji se mezcle como texto normal.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    if EMOJI_LIBRARY_AVAILABLE:
        try:
            # Convierte emojis a nombres en español:
            # 😡 -> :cara_enojada:
            text = emoji.demojize(text, language="es")
        except Exception:
            # Si por alguna razón falla español, usa inglés como respaldo.
            text = emoji.demojize(text, language="en")

        # Convierte :cara_enojada: en [emoji_cara_enojada]
        text = re.sub(
            r":([a-zA-Z0-9_+\-áéíóúñÁÉÍÓÚÑüÜ]+):",
            lambda match: " [emoji_" + match.group(1).lower().replace("-", "_") + "] ",
            text
        )

        return text

    # Si no está instalada la librería emoji, usa token genérico
    result = []

    for char in text:
        if is_emoji_char(char):
            result.append(" [emoji] ")
        else:
            result.append(char)

    return "".join(result)


def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Convertir a minúsculas
    text = text.lower()

    # Quitar URLs
    text = remove_urls(text)

    # Quitar caracteres invisibles
    text = remove_control_characters(text)

    # Quitar saltos de línea y tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    # Reducir signos repetidos
    text = reduce_repeated_punctuation(text)

    # Quitar acentos solo si se activa
    if REMOVE_ACCENTS:
        text = remove_accents_from_text(text)

    # Convertir emojis a texto
    if CONVERT_EMOJIS_TO_TEXT:
        text = convert_emojis_to_text(text)

    # Quitar espacios extra
    text = re.sub(r"\s+", " ", text)

    # Quitar espacios al inicio y final
    text = text.strip()

    return text


def count_words(text):
    if pd.isna(text):
        return 0

    text = str(text).strip()

    if text == "":
        return 0

    words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
    return len(words)


def contains_numbers(text):
    if pd.isna(text):
        return False

    return bool(re.search(r"\d", str(text)))


def convert_numeric_columns(df):
    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def clean_rows(df):
    original_rows = df.shape[0]

    empty_reviews_before = count_empty_review_text(df)

    # Eliminar filas sin texto
    df = df[df["review_text"].apply(has_valid_review_text)].copy()

    rows_after_empty_removal = df.shape[0]
    rows_removed_empty = original_rows - rows_after_empty_removal

    # Eliminar duplicados exactos
    before_exact = df.shape[0]
    df = df.drop_duplicates(keep="first").copy()
    after_exact = df.shape[0]

    exact_duplicates_removed = before_exact - after_exact

    # Eliminar duplicados por negocio + texto + rating individual
    before_subset = df.shape[0]
    df = df.drop_duplicates(
        subset=["place_id", "review_text", "review_rating"],
        keep="first"
    ).copy()
    after_subset = df.shape[0]

    subset_duplicates_removed = before_subset - after_subset

    stats = {
        "original_rows": original_rows,
        "empty_reviews_before": empty_reviews_before,
        "rows_after_empty_removal": rows_after_empty_removal,
        "rows_removed_empty": rows_removed_empty,
        "exact_duplicates_removed": exact_duplicates_removed,
        "subset_duplicates_removed": subset_duplicates_removed
    }

    return df, stats


def add_quality_columns(df):
    df["has_review_text"] = df["review_text"].apply(has_valid_review_text)

    df["has_url"] = df["review_text"].apply(contains_url)

    df["has_emoji"] = df["review_text"].apply(contains_emoji)
    df["emoji_count"] = df["review_text"].apply(count_emojis)

    df["review_text_clean"] = df["review_text"].apply(clean_text)

    df["text_length"] = df["review_text_clean"].apply(lambda x: len(str(x)))
    df["word_count"] = df["review_text_clean"].apply(count_words)

    df["is_short_review"] = df["word_count"] <= SHORT_REVIEW_WORD_THRESHOLD

    df["clean_text_is_empty"] = df["review_text_clean"].apply(
        lambda x: str(x).strip() == ""
    )

    df["has_numbers"] = df["review_text_clean"].apply(contains_numbers)

    return df


def drop_empty_clean_text_if_configured(df):
    if DROP_EMPTY_CLEAN_TEXT:
        before = df.shape[0]
        df = df[~df["clean_text_is_empty"]].copy()
        after = df.shape[0]

        print(f"Filas eliminadas porque review_text_clean quedó vacío: {before - after}")

    return df


def add_review_id(df):
    df = df.reset_index(drop=True)
    df.insert(0, "review_id", range(1, len(df) + 1))

    return df


def reorder_columns(df):
    preferred_order = [
        "review_id",
        "source_row",
        "place_id",
        "name",
        "rating",
        "user_ratings_total",
        "review_rating",
        "review_text",
        "review_text_clean",
        "has_review_text",
        "text_length",
        "word_count",
        "is_short_review",
        "clean_text_is_empty",
        "has_url",
        "has_numbers",
        "has_emoji",
        "emoji_count",
        "possible_language"
    ]

    existing_preferred_columns = [
        col for col in preferred_order
        if col in df.columns
    ]

    additional_columns = [
        col for col in df.columns
        if col not in existing_preferred_columns
    ]

    final_columns = existing_preferred_columns + additional_columns

    return df[final_columns]


def save_clean_dataset(df, output_file):
    try:
        df.to_csv(output_file, index=False, encoding=ENCODING)
        print(f"Dataset limpio guardado como: {output_file}")
        return True

    except Exception as e:
        print("Error al guardar el dataset limpio.")
        print(f"Detalle del error: {e}")
        return False


def generate_report(df_original, df_clean, output_file, report_file, row_stats):
    report_lines = []

    report_lines.append("REPORTE DE LIMPIEZA DE REVIEWS")
    report_lines.append("VERSIÓN: EMOJIS CONVERTIDOS A TEXTO")
    report_lines.append("=" * 70)
    report_lines.append(f"Fecha y hora de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"Archivo de entrada usado: {INPUT_FILE}")
    report_lines.append(f"Archivo de salida generado: {output_file}")
    report_lines.append("")

    report_lines.append("LIBRERÍA EMOJI")
    report_lines.append("-" * 70)
    report_lines.append(f"Librería emoji disponible: {EMOJI_LIBRARY_AVAILABLE}")
    if EMOJI_LIBRARY_AVAILABLE:
        report_lines.append("Los emojis fueron convertidos a nombres de texto en inglés.")
    else:
        report_lines.append("La librería emoji no estaba instalada. Los emojis se reemplazaron por [emoji].")
    report_lines.append("")

    report_lines.append("RESUMEN GENERAL")
    report_lines.append("-" * 70)
    report_lines.append(f"Número de filas originales: {df_original.shape[0]}")
    report_lines.append(f"Número de columnas originales: {df_original.shape[1]}")
    report_lines.append(f"Reviews vacíos o nulos antes de limpiar: {row_stats['empty_reviews_before']}")
    report_lines.append(f"Filas después de eliminar vacíos: {row_stats['rows_after_empty_removal']}")
    report_lines.append(f"Duplicados exactos eliminados: {row_stats['exact_duplicates_removed']}")
    report_lines.append(
        "Duplicados por place_id + review_text + review_rating eliminados: "
        f"{row_stats['subset_duplicates_removed']}"
    )
    report_lines.append(f"Número final de filas: {df_clean.shape[0]}")
    report_lines.append(f"Número final de columnas: {df_clean.shape[1]}")
    report_lines.append(f"Número de negocios únicos finales: {df_clean['place_id'].nunique(dropna=True)}")
    report_lines.append("")

    report_lines.append("CALIDAD DEL TEXTO")
    report_lines.append("-" * 70)
    report_lines.append(f"Número de reviews cortos: {df_clean['is_short_review'].sum()}")
    report_lines.append(f"Número de textos limpios vacíos: {df_clean['clean_text_is_empty'].sum()}")
    report_lines.append(f"Número de reviews con emojis: {df_clean['has_emoji'].sum()}")
    report_lines.append(f"Número total aproximado de emojis: {df_clean['emoji_count'].sum()}")
    report_lines.append(f"Número de reviews con URLs: {df_clean['has_url'].sum()}")
    report_lines.append(f"Número de reviews con números: {df_clean['has_numbers'].sum()}")
    report_lines.append("")

    report_lines.append("NULOS DESPUÉS DE LA LIMPIEZA")
    report_lines.append("-" * 70)
    null_counts = df_clean.isnull().sum()
    for col, count in null_counts.items():
        report_lines.append(f"{col}: {count}")

    report_lines.append("")

    report_lines.append("COLUMNAS FINALES")
    report_lines.append("-" * 70)
    for col in df_clean.columns:
        report_lines.append(f"- {col}")

    report_lines.append("")

    report_lines.append("CONFIGURACIÓN USADA")
    report_lines.append("-" * 70)
    report_lines.append(f"REMOVE_ACCENTS = {REMOVE_ACCENTS}")
    report_lines.append(f"KEEP_EMOJIS = {KEEP_EMOJIS}")
    report_lines.append(f"REMOVE_EMOJIS = {REMOVE_EMOJIS}")
    report_lines.append(f"CONVERT_EMOJIS_TO_TEXT = {CONVERT_EMOJIS_TO_TEXT}")
    report_lines.append(f"SHORT_REVIEW_WORD_THRESHOLD = {SHORT_REVIEW_WORD_THRESHOLD}")
    report_lines.append(f"DROP_EMPTY_CLEAN_TEXT = {DROP_EMPTY_CLEAN_TEXT}")
    report_lines.append(f"USE_TIMESTAMP_OUTPUT = {USE_TIMESTAMP_OUTPUT}")

    try:
        with open(report_file, "w", encoding="utf-8") as file:
            file.write("\n".join(report_lines))

        print(f"Reporte de limpieza guardado como: {report_file}")
        return True

    except Exception as e:
        print("Error al guardar el reporte.")
        print(f"Detalle del error: {e}")
        return False


def show_final_summary(df_clean):
    print_separator()
    print("RESUMEN FINAL")
    print_separator()

    print(f"Filas finales: {df_clean.shape[0]}")
    print(f"Columnas finales: {df_clean.shape[1]}")
    print(f"Negocios únicos finales: {df_clean['place_id'].nunique(dropna=True)}")
    print(f"Reviews cortos: {df_clean['is_short_review'].sum()}")
    print(f"Textos limpios vacíos: {df_clean['clean_text_is_empty'].sum()}")
    print(f"Reviews con emojis: {df_clean['has_emoji'].sum()}")
    print(f"Reviews con URLs: {df_clean['has_url'].sum()}")
    print(f"Reviews con números: {df_clean['has_numbers'].sum()}")

    if EMOJI_LIBRARY_AVAILABLE:
        print("\nConversión de emojis: usando librería emoji.")
    else:
        print("\nConversión de emojis: librería emoji no instalada, usando token [emoji].")

    print("\nPrimeras filas del dataset limpio:")
    print(df_clean.head())


# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================

def main():
    print_separator()
    print("INICIANDO LIMPIEZA DE REVIEWS")
    print("MODO: CONVERTIR EMOJIS A TEXTO")
    print_separator()

    if not EMOJI_LIBRARY_AVAILABLE:
        print("Advertencia: la librería 'emoji' no está instalada.")
        print("Para mejores resultados, instala con:")
        print("pip install emoji")
        print("El script seguirá funcionando, pero usará [emoji] como reemplazo.")

    output_file, report_file = generate_output_names()

    warn_if_output_exists(output_file, report_file)

    df = load_dataset(INPUT_FILE)

    if df is None:
        return

    if not validate_required_columns(df):
        return

    df_original = df.copy()

    show_initial_review(df)

    df = convert_numeric_columns(df)

    df, row_stats = clean_rows(df)

    df = add_quality_columns(df)

    df = drop_empty_clean_text_if_configured(df)

    df = add_review_id(df)

    df = reorder_columns(df)

    dataset_saved = save_clean_dataset(df, output_file)

    report_saved = generate_report(
        df_original=df_original,
        df_clean=df,
        output_file=output_file,
        report_file=report_file,
        row_stats=row_stats
    )

    show_final_summary(df)

    print_separator()

    if dataset_saved and report_saved:
        print("Proceso terminado correctamente.")
    else:
        print("Proceso terminado, pero hubo errores al guardar uno o más archivos.")


# ============================================================
# EJECUCIÓN
# ============================================================

if __name__ == "__main__":
    main()

----------------------------------------------------------------------
INICIANDO LIMPIEZA DE REVIEWS
MODO: CONVERTIR EMOJIS A TEXTO
----------------------------------------------------------------------
Archivo leído correctamente: reviews_output.csv
Validación correcta: todas las columnas obligatorias existen.
----------------------------------------------------------------------
REVISIÓN INICIAL DEL DATASET
----------------------------------------------------------------------
Número inicial de filas: 62020
Número inicial de columnas: 7

Columnas:
- source_row
- place_id
- name
- rating
- user_ratings_total
- review_text
- review_rating

Conteo de valores nulos por columna:
source_row               0
place_id                 0
name                     5
rating                  77
user_ratings_total      77
review_text           8722
review_rating            0
dtype: int64

Reviews vacíos, nulos o solo espacios: 8722
Negocios únicos usando place_id: 15567
Duplicados exactos antes de l

In [ ]:
pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.0 MB/s eta 0:00:00
